# Search Prototype - Production Pipeline

End-to-end demo of the Spottr search pipeline matching `spottr/backend` exactly:

1. SAM2 base_plus for region segmentation (same hyperparameters as `backend/app/services/ml_engine.py`)
2. SigLIP2 for image-text encoding
3. Trained scoring head (loaded from Google Drive)
4. 9-feature MLP that ranks frames

For each query, this notebook saves:
- `mock_outputs/<slug>/response.json` - the exact dict shape returned by `GET /api/search`
- `mock_outputs/<slug>/frame_<idx>.jpg` - the matched frame with bounding box drawn
- `mock_outputs/<slug>/frame_<idx>_raw.jpg` - the raw frame without overlay

Use those files to mock the backend during the demo (or just play them back from the frontend).

## 1. Install dependencies

In [ ]:
!pip install -q "open_clip_torch>=2.31.0" "timm>=1.0.15" opencv-python pillow numpy tqdm matplotlib
!pip install -q "git+https://github.com/facebookresearch/sam2.git"
!pip install -q scikit-learn==1.6.1

## 2. Mount Google Drive and route caches

Matches the path used in `03_trained_scoring_head.ipynb` (`/content/drive/MyDrive/model-cache`). Off Colab, falls back to `./data/cache`.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/model-cache")
except ImportError:
    DRIVE_ROOT = Path("./data/cache")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CACHE = DRIVE_ROOT / "models"
HF_CACHE = DRIVE_ROOT / "hf"
MODEL_CACHE.mkdir(exist_ok=True)
HF_CACHE.mkdir(exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE / "hub")

print(f"Drive root: {DRIVE_ROOT}")
print(f"SAM2 cache: {MODEL_CACHE}")
print(f"HF cache:   {HF_CACHE}")

## 3. Imports and config

In [ ]:
import json
import pickle
import re
import urllib.request

import cv2
import numpy as np
import torch
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)
MOCK_OUT_DIR = Path("./mock_outputs")
MOCK_OUT_DIR.mkdir(exist_ok=True)

FRAME_STRIDE_SECONDS = 2.0
MAX_FRAMES = 30
CROP_PAD_RATIO = 0.25
TOP_K = 5

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v"}

## 4. Download SAM2 base_plus checkpoint

Must be the same variant as `backend/app/services/ml_engine.py` and `02_pipeline_setup.ipynb`. The trained scoring head was fit on features extracted from this exact variant.

In [ ]:
SAM2_VARIANT = "sam2.1_hiera_base_plus.pt"
SAM2_CHECKPOINT_URL = f"https://dl.fbaipublicfiles.com/segment_anything_2/092824/{SAM2_VARIANT}"
SAM2_CHECKPOINT = MODEL_CACHE / SAM2_VARIANT
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_b+.yaml"

if not SAM2_CHECKPOINT.exists():
    print(f"Downloading {SAM2_CHECKPOINT_URL}")
    print(f"  -> {SAM2_CHECKPOINT}")
    urllib.request.urlretrieve(SAM2_CHECKPOINT_URL, str(SAM2_CHECKPOINT))
    print("Done.")
else:
    print(f"Cached: {SAM2_CHECKPOINT}")

## 5. Load SAM2 (backend hyperparameters)

`points_per_side=16, pred_iou_thresh=0.7` matches the training notebook. Do not change without retraining the scoring head.

In [ ]:
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

sam2_model = build_sam2(SAM2_CONFIG, str(SAM2_CHECKPOINT), device=DEVICE)
mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2_model,
    points_per_side=16,
    pred_iou_thresh=0.7,
)
print("SAM2 ready.")

## 6. Load SigLIP2

In [ ]:
import open_clip

siglip_model, siglip_preprocess = open_clip.create_model_from_pretrained(
    "hf-hub:timm/ViT-B-16-SigLIP2-256"
)
siglip_model = siglip_model.to(DEVICE).eval()
siglip_tokenizer = open_clip.get_tokenizer("hf-hub:timm/ViT-B-16-SigLIP2-256")

with torch.no_grad():
    probe = siglip_tokenizer(["probe"]).to(DEVICE)
    SIGLIP_DIM = siglip_model.encode_text(probe).shape[-1]
print(f"SigLIP2 ready. dim={SIGLIP_DIM}")


def encode_images_siglip(pil_images, batch_size=32):
    if not pil_images:
        return np.zeros((0, SIGLIP_DIM), dtype=np.float32)
    out = []
    for i in range(0, len(pil_images), batch_size):
        batch = torch.stack([siglip_preprocess(im) for im in pil_images[i:i+batch_size]]).to(DEVICE)
        with torch.no_grad():
            emb = siglip_model.encode_image(batch)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        out.append(emb.cpu().numpy())
    return np.vstack(out).astype(np.float32)


def encode_text_siglip(text):
    tokens = siglip_tokenizer([text]).to(DEVICE)
    with torch.no_grad():
        emb = siglip_model.encode_text(tokens)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().numpy().astype(np.float32)[0]

## 7. Load trained scoring head from Drive

Produced by `03_trained_scoring_head.ipynb`. The pickle contains `model` (sklearn MLPClassifier or similar), `scaler` (StandardScaler), and metadata. If missing, search falls back to the `max(region, whole)` baseline.

In [ ]:
SCORING_HEAD_PATH = DRIVE_ROOT / "trained_scoring_head.pkl"

if SCORING_HEAD_PATH.exists():
    with open(SCORING_HEAD_PATH, "rb") as f:
        trained = pickle.load(f)
    mlp_model = trained["model"]
    mlp_scaler = trained["scaler"]
    print(f"Scoring head loaded from: {SCORING_HEAD_PATH}")
    for k in ("variant", "variant_name", "f1", "metrics"):
        if k in trained:
            print(f"  {k}: {trained[k]}")
else:
    mlp_model = None
    mlp_scaler = None
    print(f"WARNING: scoring head not found at {SCORING_HEAD_PATH}")
    print("Search will fall back to max(region, whole) baseline.")

## 8. Pick input video

Edit `INPUT_PATH`. Works with images (single frame) and videos (sampled at `FRAME_STRIDE_SECONDS`, capped at `MAX_FRAMES`).

In [ ]:
INPUT_PATH = Path("./data/input.mp4")  # <-- edit to your file

ext = INPUT_PATH.suffix.lower()
assert INPUT_PATH.exists(), f"File not found: {INPUT_PATH.resolve()}"
assert ext in IMAGE_EXTS | VIDEO_EXTS, f"Unsupported extension: {ext}"
print(f"Input: {INPUT_PATH} ({INPUT_PATH.stat().st_size/1e6:.1f} MB)")

## 9. Load frames

In [ ]:
def load_image(path):
    img = Image.open(path).convert("RGB")
    return [img], [0.0], None


def load_video(path, stride_seconds=2.0, max_frames=30):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, int(round(fps * stride_seconds)))

    out_frames, ts = [], []
    for fidx in range(0, total, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
        ok, bgr = cap.read()
        if not ok:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        out_frames.append(Image.fromarray(rgb))
        ts.append(fidx / fps)
        if len(out_frames) >= max_frames:
            break
    cap.release()
    return out_frames, ts, fps


def load_frames(path):
    e = Path(path).suffix.lower()
    if e in IMAGE_EXTS:
        return load_image(path)
    return load_video(path, FRAME_STRIDE_SECONDS, MAX_FRAMES)


frames, timestamps, fps = load_frames(INPUT_PATH)
kind = "image" if fps is None else f"video @ {fps:.1f} fps"
print(f"Loaded {len(frames)} frame(s) from {INPUT_PATH.name} ({kind})")

## 10. Build the index (matches backend `video_processor.py`)

Key invariants the search code depends on:
- The **last** region per frame is the whole frame.
- `region_areas` and `region_ious` are tracked because the scoring head's 9 features include them.

In [ ]:
def crops_and_boxes_from_masks(pil_image, masks, pad_ratio=CROP_PAD_RATIO, min_side=10):
    arr = np.array(pil_image)
    H, W = arr.shape[:2]
    crops, boxes, areas, ious = [], [], [], []
    for m in masks:
        x, y, w, h = [int(v) for v in m["bbox"]]
        pad_w, pad_h = int(w * pad_ratio), int(h * pad_ratio)
        x1, y1 = max(0, x - pad_w), max(0, y - pad_h)
        x2, y2 = min(W, x + w + pad_w), min(H, y + h + pad_h)
        if x2 - x1 < min_side or y2 - y1 < min_side:
            continue
        crops.append(Image.fromarray(arr[y1:y2, x1:x2]))
        boxes.append((x1, y1, x2 - x1, y2 - y1))
        areas.append(m["area"] / (W * H))
        ious.append(m["predicted_iou"])
    crops.append(pil_image)
    boxes.append((0, 0, W, H))
    areas.append(1.0)
    ious.append(1.0)
    return crops, boxes, areas, ious


all_embeddings = []
region_to_frame = []
region_boxes = []
region_areas = []
region_ious = []

for f_idx, frame in enumerate(tqdm(frames, desc="Indexing")):
    masks = mask_generator.generate(np.array(frame))
    crops, boxes, areas, ious = crops_and_boxes_from_masks(frame, masks)
    embs = encode_images_siglip(crops)
    all_embeddings.append(embs)
    region_to_frame.extend([f_idx] * embs.shape[0])
    region_boxes.extend(boxes)
    region_areas.extend(areas)
    region_ious.extend(ious)

if all_embeddings:
    index = np.vstack(all_embeddings).astype(np.float32)
else:
    index = np.zeros((0, SIGLIP_DIM), dtype=np.float32)
region_to_frame = np.array(region_to_frame, dtype=np.int64)
region_boxes = np.array(region_boxes, dtype=np.int64)
region_areas = np.array(region_areas, dtype=np.float32)
region_ious = np.array(region_ious, dtype=np.float32)

print(f"Index: {index.shape[0]} regions across {len(frames)} frame(s) "
      f"({index.shape[0]/max(1,len(frames)):.1f} regions/frame)")

## 11. Search (matches backend `search.py`)

Builds the 9-feature vector per frame, runs the trained MLP, falls back to `max(region, whole)` if the head is missing. Returns a dict in the exact shape of `GET /api/search`.

In [ ]:
def extract_features_per_frame(sims):
    num_frames = len(frames)
    feats = np.zeros((num_frames, 9), dtype=np.float32)
    best_score = np.full(num_frames, -np.inf, dtype=np.float32)
    best_region = np.full(num_frames, -1, dtype=np.int64)

    for frame_idx in range(num_frames):
        idxs = np.where(region_to_frame == frame_idx)[0]
        if len(idxs) == 0:
            continue
        rs = sims[idxs]
        whole = float(rs[-1])
        local_rs = rs[:-1]

        if len(local_rs) > 0:
            best_local_idx = int(local_rs.argmax())
            best = float(local_rs[best_local_idx])
            mean_top3 = float(np.sort(local_rs)[-3:].mean())
            std_r = float(local_rs.std()) if len(local_rs) > 1 else 0.0
            num_r = int(len(local_rs))
            best_area = float(region_areas[idxs[best_local_idx]])
            best_iou = float(region_ious[idxs[best_local_idx]])
            best_score[frame_idx] = max(best, whole)
            best_region[frame_idx] = idxs[best_local_idx] if best >= whole else idxs[-1]
        else:
            best, mean_top3, std_r, num_r, best_area, best_iou = whole, whole, 0.0, 0, 1.0, 1.0
            best_score[frame_idx] = whole
            best_region[frame_idx] = idxs[-1]

        feats[frame_idx] = [whole, best, mean_top3, std_r, num_r, best_area, best_iou, best - whole, max(best, whole)]
    return feats, best_score, best_region


def search(query, top_k=TOP_K):
    text_emb = encode_text_siglip(query)
    sims = index @ text_emb
    feats, best_score, best_region = extract_features_per_frame(sims)

    if mlp_model is not None and mlp_scaler is not None:
        feats_scaled = mlp_scaler.transform(feats)
        probs = mlp_model.predict_proba(feats_scaled)[:, 1]
        order = np.argsort(-probs)[:top_k]
        scores_to_return = probs
        method = "mlp"
    else:
        order = np.argsort(-best_score)[:top_k]
        scores_to_return = best_score
        method = "baseline"

    results = []
    for i in order:
        frame_idx = int(i)
        if best_score[frame_idx] == -np.inf:
            continue
        region_idx = int(best_region[frame_idx])
        x, y, w, h = map(int, region_boxes[region_idx])
        results.append({
            "frame_idx": frame_idx,
            "timestamp": float(timestamps[frame_idx]),
            "score": float(scores_to_return[frame_idx]),
            "box": {"x": x, "y": y, "w": w, "h": h},
        })

    return {"query": query, "results": results, "scoring_method": method}

## 12. Display

In [ ]:
def show_results(response):
    hits = response["results"]
    n = len(hits)
    if n == 0:
        print("No results.")
        return
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, hit in zip(axes, hits):
        frame = frames[hit["frame_idx"]]
        ax.imshow(frame)
        b = hit["box"]
        is_whole = (b["w"] >= frame.size[0] and b["h"] >= frame.size[1])
        ax.add_patch(patches.Rectangle(
            (b["x"], b["y"]), b["w"], b["h"],
            fill=False, edgecolor="yellow" if is_whole else "lime", linewidth=2.5,
        ))
        title = f"frame {hit['frame_idx']}"
        if fps is not None:
            title += f" @ {hit['timestamp']:.1f}s"
        title += f"\nscore={hit['score']:.3f}"
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(f'query: "{response["query"]}"  ({response["scoring_method"]})', fontsize=13)
    plt.tight_layout()
    plt.show()

## 13. Run mock queries and save outputs

For each query, writes a folder under `mock_outputs/`:
- `response.json` - exact GET /api/search response
- `frame_<idx>.jpg` - matched frame with bounding box drawn
- `frame_<idx>_raw.jpg` - matched frame without overlay

During the demo you can either point the frontend at this folder, or just open the JSON in any HTTP mocking tool.

In [ ]:
def slug(query):
    return re.sub(r"[^a-z0-9]+", "_", query.lower()).strip("_")[:50]


def save_frame_with_box(frame, box, dest):
    img = frame.copy()
    draw = ImageDraw.Draw(img)
    draw.rectangle(
        [box["x"], box["y"], box["x"] + box["w"], box["y"] + box["h"]],
        outline="lime", width=4,
    )
    img.save(dest, "JPEG", quality=90)


def save_mock(response):
    out_dir = MOCK_OUT_DIR / slug(response["query"])
    out_dir.mkdir(parents=True, exist_ok=True)

    with open(out_dir / "response.json", "w") as f:
        json.dump(response, f, indent=2)

    for hit in response["results"]:
        frame = frames[hit["frame_idx"]]
        save_frame_with_box(frame, hit["box"], out_dir / f"frame_{hit['frame_idx']}.jpg")
        frame.save(out_dir / f"frame_{hit['frame_idx']}_raw.jpg", "JPEG", quality=90)

    return out_dir


QUERIES = [
    "a person walking",
    "a red car",
    "a dog",
]

for q in QUERIES:
    print(f"\n=== {q} ===")
    response = search(q, top_k=TOP_K)
    show_results(response)
    out_dir = save_mock(response)
    print(f"Saved -> {out_dir}")
    print(json.dumps(response, indent=2))

## 14. Inspect saved outputs

Quick sanity check: list everything we wrote and show one frame back from disk.

In [ ]:
for d in sorted(MOCK_OUT_DIR.iterdir()):
    if not d.is_dir():
        continue
    files = sorted(p.name for p in d.iterdir())
    print(f"{d.name}/")
    for name in files:
        size_kb = (d / name).stat().st_size / 1024
        print(f"  {name}  ({size_kb:.1f} KB)")